In [0]:

dbutils.widgets.text("login", "")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("container", "")
dbutils.widgets.text("secret_scope", "")

In [0]:
login = dbutils.widgets.get("login")
storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")
secret_scope = dbutils.widgets.get("secret_scope")

mount_point = f"/mnt/{container}"
source = f"abfss://{container}@{storage_account}.dfs.core.windows.net/"

In [0]:
dbutils.secrets.list(secret_scope)

In [0]:
try:
    client_id = dbutils.secrets.get(scope=secret_scope, key="sp-databricks-adls-appid")
    client_secret = dbutils.secrets.get(scope=secret_scope, key="sp-databricks-adls-appkey")
    tenant_id = dbutils.secrets.get(scope=secret_scope, key="tenant-id")

    configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}
    
    if any(mount.mountPoint == mount_point for mount in dbutils.fs.mounts()):
        print(f"Mount already exists: {mount_point}")
    else: 
        dbutils.fs.mount(
            source = source,
            mount_point = mount_point,
            extra_configs = configs)
        print(f"Mount was created: {mount_point}")
    display(dbutils.fs.ls(mount_point))
except Exception as e:
    print(e)

The mount couldn't be created because DBFS root and mounts are disabled.